# Skill Boundary Audit v1

本实验验证 `src/core` 新增的 skill boundary audit：skill 只保留方法论提示，具体项目路径、运行命令、模型结果和业务入口应迁回 notebook、`src/` 或 manifest。

In [1]:
from pathlib import Path
import json
import sys

current = Path.cwd().resolve()
repo_root = next(candidate for candidate in (current, *current.parents) if (candidate / "src").is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.core import build_skill_boundary_audit

OUTPUT_DIR = repo_root / "research" / "skill_boundary_audit_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 样本

一个样本代表 method-only skill，只提示研究思路；另一个样本代表需要迁移的旧 workflow skill，包含具体 runtime 路径、旧业务入口、固定命令和模型结果。

In [2]:
method_only_skill = """
# Regime Review Method

Use this skill to brainstorm candidate market regimes, ask whether factor
evidence should be split by volatility and drawdown, and remind the model to
turn any useful idea into a research notebook before implementation.

Keep concrete data paths, model results, fixed parameters, and project state
outside the skill.
"""

project_state_skill = """
# SOXL Qlib Experiment Workflow

Put each experiment under runtime/model_selection/card/rolling_selector_v1/.
Import framework APIs from soxl_backtest.framework and run:

PYTHONPATH=. qlib_venv/bin/python -m pytest tests/test_soxlbacktest_framework.py -q

Current best model: ridge_mse sharpe=1.42 max_drawdown=12.5%.
"""

method_audit = build_skill_boundary_audit("regime-method", method_only_skill)
project_state_audit = build_skill_boundary_audit("soxl-qlib-experiment", project_state_skill)

method_audit, project_state_audit

({'schema_version': 'core.skill_boundary_audit.v1',
  'skill_name': 'regime-method',
  'source_path': '',
  'status': 'pass',
  'method_only': True,
  'allowed_skill_role': 'method_prompt',
  'issue_count': 0,
  'blocked_issue_count': 0,
  'review_issue_count': 0,
  'migration_required': False,
  'recommended_destination': 'keep_in_skill',
  'blocked_surfaces_before_migration': [],
  'issues': []},
 {'schema_version': 'core.skill_boundary_audit.v1',
  'skill_name': 'soxl-qlib-experiment',
  'source_path': '',
  'status': 'block',
  'method_only': False,
  'allowed_skill_role': 'method_prompt',
  'issue_count': 7,
  'blocked_issue_count': 7,
  'review_issue_count': 0,
  'migration_required': True,
  'recommended_destination': 'research_notebook_or_src_core_manifest',
  'blocked_surfaces_before_migration': ['skill'],
  'issues': [{'schema_version': 'core.skill_boundary_issue.v1',
    'issue_type': 'concrete_project_path',
    'severity': 'block',
    'line': 4,
    'matched_text': 'runti

## 断言

method-only 样本必须通过；包含项目状态和业务入口的样本必须阻断，并给出迁移建议。

In [3]:
assert method_audit["schema_version"] == "core.skill_boundary_audit.v1"
assert method_audit["status"] == "pass"
assert method_audit["method_only"] is True
assert method_audit["issue_count"] == 0

assert project_state_audit["status"] == "block"
assert project_state_audit["method_only"] is False
assert project_state_audit["migration_required"] is True
assert project_state_audit["recommended_destination"] == "research_notebook_or_src_core_manifest"
issue_types = {issue["issue_type"] for issue in project_state_audit["issues"]}
assert {
    "concrete_project_path",
    "executable_command",
    "business_logic_entrypoint",
    "project_state_result",
}.issubset(issue_types)
assert all(issue["recommendation"] for issue in project_state_audit["issues"])

In [4]:
analysis_summary = {
    "schema_version": "research.skill_boundary_audit_v1.summary.v1",
    "experiment": "skill_boundary_audit_v1",
    "core_contract": "core.skill_boundary_audit.v1",
    "method_only_status": method_audit["status"],
    "method_only_issue_count": method_audit["issue_count"],
    "project_state_status": project_state_audit["status"],
    "project_state_issue_count": project_state_audit["issue_count"],
    "project_state_blocked_issue_count": project_state_audit["blocked_issue_count"],
    "project_state_issue_types": sorted(issue_types),
    "promotion_decision": "promote_to_src_core_contract",
    "risk_note": "静态审计只识别常见边界问题，不能替代人工 review skill 语义。",
}

summary_path = OUTPUT_DIR / "analysis_summary.json"
summary_path.write_text(json.dumps(analysis_summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
analysis_summary

{'schema_version': 'research.skill_boundary_audit_v1.summary.v1',
 'experiment': 'skill_boundary_audit_v1',
 'core_contract': 'core.skill_boundary_audit.v1',
 'method_only_status': 'pass',
 'method_only_issue_count': 0,
 'project_state_status': 'block',
 'project_state_issue_count': 7,
 'project_state_blocked_issue_count': 7,
 'project_state_issue_types': ['business_logic_entrypoint',
  'concrete_project_path',
  'executable_command',
  'project_state_result'],
 'promotion_decision': 'promote_to_src_core_contract',
 'risk_note': '静态审计只识别常见边界问题，不能替代人工 review skill 语义。'}

## 中文结论

- 做了什么：新增并验证 `core.skill_boundary_audit.v1`，把 skill 边界从口头约定变成可测试契约；method-only skill 通过，包含具体项目路径、命令、旧业务入口和模型结果的 skill 被阻断。
- 核心指标：method-only 样本 `status=pass`、`issue_count=0`；项目状态样本 `status=block`，覆盖 `concrete_project_path`、`executable_command`、`business_logic_entrypoint`、`project_state_result` 四类问题。
- regime 差异：本实验不是市场 regime 回测，而是协作/知识治理 regime；方法论文本可留在 skill，项目状态和可运行功能必须迁回 notebook、`src/` 或 manifest。
- 是否晋升：晋升到 `src/core`，因为 skill 边界是跨研究、production、dashboard 和 schedule 的稳定治理契约。
- 风险：当前是静态文本审计，能发现常见路径、命令、入口和结果污染，但不能替代人工判断 skill 是否过度具体。
- 下一步：将该 audit 接入 notebook-card 或知识 ingest review，持续发现需要从 skill 迁移回框架的旧流程。